### Emily Dickinson prep

#### Install packages

In [1]:
# ===== Install-if-missing utilities (Python only) =====
import sys, importlib, subprocess

def _pip_install(pkg, *extra_args):
    """Run pip for this interpreter."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, *extra_args])

def ensure_packages(packages, upgrade=False, user=False, quiet=True):
    """
    Ensure each package in `packages` is importable.
    Installs only the missing ones. Options:
      - upgrade=True  -> add '--upgrade'
      - user=True     -> add '--user' (helpful if no venv / no write perms)
      - quiet=True    -> add '-q' to reduce output noise
    """
    extra = []
    if upgrade: extra.append("--upgrade")
    if user:    extra.append("--user")
    if quiet:   extra.append("-q")
    for name in packages:
        try:
            importlib.import_module(name if name != "sentence-transformers" else "sentence_transformers")
        except ImportError:
            _pip_install(name, *extra)

def ensure_spacy_model(model="en_core_web_sm"):
    """
    Load a spaCy model, downloading if missing.
    """
    try:
        import spacy; spacy.load(model)
    except Exception:
        # try spacy's built-in downloader first
        try:
            import spacy.cli as sc; sc.download(model)
        except Exception:
            # fallback: install as a pip package (some envs prefer this)
            _pip_install(model, "-q")



#### Prep corpus

In [ ]:
# Emily Dickinson — parse Gutenberg corpus, preserve line breaks,
# remove bracketed editorial notes, and export CSV + TXT.

import re
import csv
from pathlib import Path
import pandas as pd

# ---- 1) Paths ----
SRC = Path("emily.txt")  # your uploaded file
OUT_CSV = Path("emily_prepared.csv")
OUT_TXT = Path("emily_prepared.txt")

# ---- 2) Read (Gutenberg file is ISO-8859-1 / latin-1) ----
raw = SRC.read_text(encoding="latin-1", errors="ignore")

# ---- 3) Trim Gutenberg boilerplate (keep only the ebook body) ----
start_marker = "*** START OF THIS PROJECT GUTENBERG EBOOK"
end_marker   = "*** END OF THIS PROJECT GUTENBERG EBOOK"
sidx = raw.find(start_marker)
eidx = raw.find(end_marker)
if sidx != -1 and eidx != -1:
    raw = raw[sidx + len(start_marker) : eidx]

# ---- 4) Remove bracketed editorial notes (e.g., [Published in ...]) ----
# Uses DOTALL so notes spanning multiple lines are removed.
raw = re.sub(r"\[.*?\]", "", raw, flags=re.DOTALL)

# Optional: collapse runs of 3+ blank lines to exactly 2,
# to keep spacing readable after note removals (comment out if not desired).
raw = re.sub(r"(?:\n\s*){3,}", "\n\n", raw)

# ---- 5) Split poems by Roman numeral headers at the start of blocks ----
# Matches blankline(s) + ROMAN. + newline on its own line (I., II., III., …)
blocks = re.split(r"\n\s*\n[IVXLCDM]+\.\s*\n", raw)

# The first split chunk is front matter (preface/contents). Drop it.
blocks = [b for b in (blk.strip("\n") for blk in blocks) if b.strip()]
if blocks:
    blocks = blocks[1:]  # drop preface chunk

poems = []
for i, block in enumerate(blocks, start=1):
    lines = [ln.rstrip() for ln in block.splitlines()]
    lines = [ln for ln in lines if ln.strip()]  # drop empty lines at edges

    # Identify a TITLE line (full uppercase, >2 chars, usually near top)
    title_idx = None
    for idx, ln in enumerate(lines[:6]):  # check first few lines
        if ln.strip().isupper() and len(ln.strip()) > 2:
            title_idx = idx
            break

    if title_idx is not None:
        title = lines[title_idx].strip()
        text_lines = lines[title_idx + 1 :]
    else:
        # 🔑 Use first non-empty line of the poem as the "title"
        title = lines[0].strip() if lines else f"Poem_{i}"
        text_lines = lines[1:]

    text = "\n".join(text_lines).strip("\n")

    poems.append({"id": i, "title": title, "text": text})


# ---- 6) Export CSV (newlines preserved inside the "text" field) ----
df = pd.DataFrame(poems, columns=["id", "title", "text"])
# Ensure embedded newlines are properly quoted
df.to_csv(OUT_CSV, index=False, encoding="utf-8", lineterminator="\n", quoting=csv.QUOTE_MINIMAL)

# ---- 7) Export pretty TXT anthology with titles and blank lines preserved ----
with OUT_TXT.open("w", encoding="utf-8", newline="\n") as f:
    for row in poems:
        f.write(f"{row['title']}\n\n")
        f.write(f"{row['text']}\n\n")  # blank line between poems

print(f"✅ Parsed {len(poems)} poems")
print(f"CSV: {OUT_CSV.resolve()}")
print(f"TXT: {OUT_TXT.resolve()}")


✅ Parsed 448 poems
CSV: C:\Users\dkill\OneDrive\Documents\Poetry\Emily Dickinson\emily_prepared.csv
TXT: C:\Users\dkill\OneDrive\Documents\Poetry\Emily Dickinson\emily_prepared.txt


In [5]:
import re
import pandas as pd

# Path to your uploaded file
corpus_path = "emily.txt"

# Read the raw text
with open(corpus_path, "r", encoding="latin-1") as f:
    raw_text = f.read()

# Remove Gutenberg boilerplate (everything before "*** START" and after "*** END")
start_marker = "*** START OF THIS PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THIS PROJECT GUTENBERG EBOOK"

start_idx = raw_text.find(start_marker)
end_idx = raw_text.find(end_marker)

if start_idx != -1 and end_idx != -1:
    raw_text = raw_text[start_idx+len(start_marker):end_idx]

# Remove bracketed editorial notes like [Published in ...] ---
raw_text = re.sub(r"\[.*?\]", "", raw_text, flags=re.DOTALL)

# Split poems: Dickinson’s edition numbers poems (I., II., III., etc.)
# We'll use regex to split on Roman numerals at the start of lines
poem_splits = re.split(r"\n\n+[IVXLCDM]+\.\s*\n", raw_text)

# First chunk will be prefatory matter — drop it
poems_raw = [p.strip() for p in poem_splits if len(p.strip()) > 0][1:]

# Extract poem titles (in all caps, usually on their own line) and text
poems = []
for i, block in enumerate(poems_raw, start=1):
    lines = block.splitlines()
    title = None
    text_lines = []

    for line in lines:
        if line.strip().isupper() and len(line.strip()) > 2:
            # Treat as title
            title = line.strip()
        else:
            text_lines.append(line.strip())

    text = " ".join([l for l in text_lines if l])  # join non-empty lines

    poems.append({
        "id": i,
        "title": title if title else f"Untitled_{i}",
        "text": text
    })

# Create DataFrame
df_poems = pd.DataFrame(poems)

# Save to CSV if needed
df_poems.to_csv("emily_prepared.txt", index=False, encoding="utf-8")

print(df_poems.head(10))


   id           title                                               text
0   1        SUCCESS.  Success is counted sweetest By those who ne'er...
1   2      Untitled_2  Our share of night to bear, Our share of morni...
2   3  ROUGE ET NOIR.  Soul, wilt thou toss again? By just such a haz...
3   4    ROUGE GAGNE.  'T is so much joy! 'T is so much joy! If I sho...
4   5      Untitled_5  Glee! The great storm is over! Four have recov...
5   6      Untitled_6  If I can stop one heart from breaking, I shall...
6   7         ALMOST!  Within my reach! I could have touched! I might...
7   8      Untitled_8  A wounded deer leaps highest, I've heard the h...
8   9      Untitled_9  The heart asks pleasure first, And then, excus...
9  10   IN A LIBRARY.  A precious, mouldering pleasure 't is To meet ...


In [3]:
# --- Task 1: Corpus Preparation (Emily Dickinson) ----------------------------
# - Accepts either:
#     (A) a single .txt file containing many poems separated by blank lines
#     (B) a folder of .txt files (each file = one poem; filename = title)
# - Normalizes encoding (NFKC) and unifies dash glyphs.
# - Produces df[id, title, text] and writes CSV.

from pathlib import Path
import re, unicodedata
import pandas as pd

# >>>> SET THIS TO YOUR SOURCE <<<<
INPUT_PATH = Path("emily.txt")  # or Path("path/to/folder_of_poems")
OUTPUT_CSV = Path("emily_clean.csv")

def normalize_text(s: str) -> str:
    """Light normalization that preserves Dickinson’s style (dashes kept)."""
    # unify common dash glyphs to en dash (–); you can switch to em dash if preferred
    s = s.replace("—", "–").replace("-", "–")
    # Unicode normalization
    s = unicodedata.normalize("NFKC", s)
    # strip trailing spaces on lines
    s = "\n".join(line.rstrip() for line in s.splitlines())
    return s

def strip_gutenberg_boilerplate(s: str) -> str:
    """Optionally remove Project Gutenberg boilerplate if present."""
    start_pat = r"\*\*\*\s*START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\n"
    end_pat   = r"\n\*\*\*\s*END OF (THE|THIS) PROJECT GUTENBERG EBOOK.*"
    s = re.sub(start_pat, "", s, flags=re.IGNORECASE|re.DOTALL)
    s = re.sub(end_pat, "", s, flags=re.IGNORECASE|re.DOTALL)
    return s.strip()

def parse_single_txt_file(path: Path) -> pd.DataFrame:
    """Parse one big .txt into poems by blank lines; first non-empty line = title."""
    raw = path.read_text(encoding="utf-8", errors="ignore")
    raw = strip_gutenberg_boilerplate(raw)
    raw = normalize_text(raw)

    # split on blank lines; keep blocks with at least one letter
    blocks = [b.strip() for b in re.split(r"\n\s*\n", raw) if re.search(r"[A-Za-z]", b)]
    records = []
    for i, block in enumerate(blocks, 1):
        lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
        if not lines:
            continue
        # If only one line, treat it as the poem text and auto-title
        if len(lines) == 1:
            title = f"Poem {i}"
            text  = lines[0]
        else:
            title = lines[0]
            text  = "\n".join(lines[1:])
        records.append({"id": i, "title": title, "text": text})
    return pd.DataFrame.from_records(records, columns=["id","title","text"])

def parse_folder_of_txts(folder: Path) -> pd.DataFrame:
    """Each .txt file becomes a poem; filename (sans extension) as title."""
    records = []
    for i, f in enumerate(sorted(folder.glob("*.txt")), 1):
        txt = f.read_text(encoding="utf-8", errors="ignore")
        txt = strip_gutenberg_boilerplate(txt)
        txt = normalize_text(txt).strip()
        # If the first line is actually a title line duplicated in text, you can drop it here if needed.
        title = f.stem
        records.append({"id": i, "title": title, "text": txt})
    return pd.DataFrame.from_records(records, columns=["id","title","text"])

def build_corpus(input_path: Path) -> pd.DataFrame:
    if input_path.is_dir():
        df = parse_folder_of_txts(input_path)
    elif input_path.is_file():
        df = parse_single_txt_file(input_path)
    else:
        raise FileNotFoundError(f"Input not found: {input_path}")
    # final tidy: collapse repeated blank lines inside poem text to single blank line
    df["text"] = df["text"].map(lambda s: re.sub(r"(?:\n\s*){3,}", "\n\n", s).strip())
    # ensure simple integer ids starting at 1
    df = df.sort_values("id").reset_index(drop=True)
    df["id"] = range(1, len(df) + 1)
    return df

if __name__ == "__main__":
    df = build_corpus(INPUT_PATH)
    print(f"Parsed {len(df)} poems.")
    # quick peek
    print(df.head(3).to_string(index=False))
    # save
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"Saved corpus to {OUTPUT_CSV.resolve()}")


Parsed 2003 poems.
 id                                                            title                                                                                                                                                                                      text
  1                                                           Poem 1                                                                                                                     Project Gutenberg's Poems: Three Series, Complete, by Emily Dickinson
  2 This eBook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever.  You may copy it, give it away or\nre–use it under the terms of the Project Gutenberg License included\nwith this eBook or online at www.gutenberg.org
  3                                                           Poem 3                                                                                                                                                    